## Install Libraries

In [44]:
!pip install transformers seqeval evaluate conllu accelerate datasets -q

# Import Libraries

In [45]:
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from evaluate import load
from conllu import parse

# Download Universal Dependencies Dataset

In [46]:
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu

--2026-04-05 06:17:22--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15029817 (14M) [text/plain]
Saving to: ‘en_ewt-ud-train.conllu.1’

en_ewt-ud-train.con 100%[===================>]  14.33M  --.-KB/s    in 0.05s   

2026-04-05 06:17:22 (264 MB/s) - ‘en_ewt-ud-train.conllu.1’ saved [15029817/15029817]

--2026-04-05 06:17:22--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP reque

# Load Dataset

In [47]:
def load_ud_data(file_path):

    tokens_list = []
    pos_tags_list = []

    with open(file_path, "r", encoding="utf-8") as f:
        sentences = parse(f.read())

    for sentence in sentences:

        tokens = []
        pos_tags = []

        for token in sentence:
            tokens.append(token["form"])
            pos_tags.append(token["upos"])

        tokens_list.append(tokens)
        pos_tags_list.append(pos_tags)

    return tokens_list, pos_tags_list

# Prepare Train and Test Data

In [48]:
train_tokens, train_pos = load_ud_data("en_ewt-ud-train.conllu")
test_tokens, test_pos = load_ud_data("en_ewt-ud-dev.conllu")

# Use only first 4000 samples
train_tokens = train_tokens[:4000]
train_pos = train_pos[:4000]

test_tokens = test_tokens[:500]
test_pos = test_pos[:500]

print(train_tokens[0])
print(train_pos[0])

['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
['PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'PROPN', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'PROPN', 'PUNCT', 'ADP', 'DET', 'ADJ', 'NOUN', 'PUNCT']


# Create Label Mapping

In [49]:
labels = sorted(list(set(tag for sent in train_pos for tag in sent)))

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

num_labels = len(labels)

print(labels)

['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X', '_']


# Load BERT Tokenizer

In [50]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenization + Label Alignment

In [51]:
def tokenize_and_align(tokens, labels):

    tokenized = tokenizer(
        tokens,
        truncation=True,
        is_split_into_words=True
    )

    word_ids = tokenized.word_ids()

    label_ids = []
    prev = None

    for word_idx in word_ids:

        if word_idx is None:
            label_ids.append(-100)

        elif word_idx != prev:
            label_ids.append(label2id[labels[word_idx]])

        else:
            label_ids.append(-100)

        prev = word_idx

    tokenized["labels"] = label_ids

    return tokenized

# Preprocess

In [52]:
train_encodings = [tokenize_and_align(t, l) for t,l in zip(train_tokens, train_pos)]
test_encodings = [tokenize_and_align(t, l) for t,l in zip(test_tokens, test_pos)]

# Convert to PyTorch Dataset

In [53]:
class TokenDataset(torch.utils.data.Dataset):

    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):

        item = {k: torch.tensor(v) for k,v in self.encodings[idx].items()}
        return item

    def __len__(self):
        return len(self.encodings)


train_dataset = TokenDataset(train_encodings)
test_dataset = TokenDataset(test_encodings)

# Model Setup

In [54]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

# Data Collator

In [55]:
data_collator = DataCollatorForTokenClassification(tokenizer)

# Compute Metrics

In [56]:
def compute_metrics(p):

    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    true_labels = [
        [id2label[l] for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Training Configuration

In [57]:
training_args = TrainingArguments(

    output_dir="./pos_model",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100
)

# Trainer

In [58]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# Train model

In [59]:
trainer.train()

Step,Training Loss
100,1.323591
200,0.282864
300,0.179971
400,0.132384
500,0.111971
600,0.086338
700,0.077559


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=0.2976026792526245, metrics={'train_runtime': 179.4074, 'train_samples_per_second': 66.887, 'train_steps_per_second': 4.18, 'total_flos': 344112274801152.0, 'train_loss': 0.2976026792526245, 'epoch': 3.0})

# Evaluation Metric

In [60]:
from evaluate import load

metric = load("seqeval")

# Evaluate

In [61]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

{'eval_loss': 0.11435813456773758,
 'eval_precision': 0.9659438420906311,
 'eval_recall': 0.964736915174233,
 'eval_f1': 0.9653400013891783,
 'eval_accuracy': 0.9708662436876861,
 'eval_runtime': 2.02,
 'eval_samples_per_second': 247.526,
 'eval_steps_per_second': 15.842,
 'epoch': 3.0}

# Inference 1

In [62]:
sentence = "John works at Google in California"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(sentence, return_tensors="pt")

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

pred_labels = [id2label[p.item()] for p in predictions[0]]

for token, label in zip(tokens, pred_labels):
    print(token, label)

[CLS] PUNCT
john PROPN
works VERB
at ADP
google PROPN
in ADP
california PROPN
[SEP] PUNCT


# Inference 2

In [63]:
sentence = "im very good at coding i will be winning this hackathon"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(sentence, return_tensors="pt")

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

pred_labels = [id2label[p.item()] for p in predictions[0]]

for token, label in zip(tokens, pred_labels):
    print(token, label)

[CLS] PUNCT
im PRON
very ADV
good ADJ
at ADP
coding NOUN
i PRON
will AUX
be AUX
winning VERB
this DET
hack NOUN
##ath NOUN
##on NOUN
[SEP] PUNCT


# Comparison POS tagging vs Chunking

| Feature    | POS Tagging                   | Chunking                 |
| ---------- | ----------------------------- | ------------------------ |
| Level      | Word-level                    | Phrase-level             |
| Goal       | Identify grammatical category | Identify phrases         |
| Example    | Noun, Verb, Adjective         | Noun Phrase, Verb Phrase |
| Difficulty | Easy                          | Medium                   |
| Dataset    | Universal Dependencies        | CoNLL-2000               |


Example

**Sentence:  The quick brown fox jumps over the lazy dog**

**POS** **tagging**:
The → DET,
quick → ADJ,
brown → ADJ,
fox → NOUN,
jumps → VERB,
over → ADP,
the → DET,
lazy → ADJ,
dog → NOUN

**Chunking**:
[The quick brown fox] → NP,
[jumps] → VP,
[over the lazy dog] → PP

**Chunking → Phrase-level grouping**

Chunking groups words into syntactic phrases such as noun phrases and verb phrases.
Unlike POS tagging which labels individual words, chunking identifies meaningful word groups.
This task is more complex because the model must learn relationships between neighboring tokens.